# Supervisor Agent 테스트 (Production Level)

수퍼바이저 에이전트 (Level 0) 테스트 노트북

## 역할: "Pipeline Orchestrator" (파이프라인 오케스트레이터)
- 전체 멀티에이전트 파이프라인 조율
- **trace_id**: 전역 추적 ID로 요청 전체 추적
- **supervisor_state**: 재시도 횟수 및 상태 모니터링
- **human_review**: 최대 재시도 초과 시 사람 개입 요청

---

### 💡 라우팅 로직

| 조건 | 다음 단계 |
|------|----------|
| extraction_done = False | parallel_extraction |
| analysis_done = False | parallel_analysis |
| validation_done = False | validation |
| validation.action == retry_extraction | parallel_extraction (재시도) |
| retry_count >= max_retries | human_review |
| 모두 완료 | __end__ |

In [1]:
import sys, os, json, asyncio
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path: sys.path.insert(0, project_root)
from dotenv import load_dotenv
load_dotenv(os.path.join(project_root, '.env'))
print(f"Project root: {project_root}")

Project root: c:\jungle\weapon\sto-link-AI-backend


In [2]:
from app.agents.supervisor import (
    supervisor_router, supervisor_node, 
    get_phase_status, get_routing_summary, get_supervisor_state,
    generate_trace_id, human_review_node,
    MAX_EXTRACTION_RETRIES, MAX_ANALYSIS_RETRIES
)

def run_async(coro):
    try:
        loop = asyncio.get_event_loop()
        if loop.is_running():
            import nest_asyncio; nest_asyncio.apply()
            return loop.run_until_complete(coro)
        return asyncio.run(coro)
    except: return asyncio.run(coro)

## 1. Trace ID 생성 테스트

In [3]:
print("="*70)
print("📋 Test 1: Trace ID 생성")
print("="*70)

trace_id = generate_trace_id()
print(f"Generated trace_id: {trace_id}")

assert trace_id.startswith("req-"), "trace_id는 'req-'로 시작해야 함"
assert len(trace_id) > 20, "trace_id는 충분히 길어야 함"
print("✅ PASS")

📋 Test 1: Trace ID 생성
Generated trace_id: req-20251228-124526-e051199b
✅ PASS


## 2. Supervisor State 구조

In [4]:
print("="*70)
print("📋 Test 2: Supervisor State 구조")
print("="*70)

test_state = {
    "trace_id": "req-test-123",
    "extraction_done": True,
    "analysis_done": False,
    "extraction_retry_count": 1
}

supervisor_state = get_supervisor_state(test_state)
print(json.dumps(supervisor_state, indent=2))

assert supervisor_state["trace_id"] == "req-test-123"
assert supervisor_state["current_phase"] == "analysis"
assert supervisor_state["retry_counts"]["extraction"] == 1
assert supervisor_state["max_retries"]["extraction"] == MAX_EXTRACTION_RETRIES
print("✅ PASS")

📋 Test 2: Supervisor State 구조
{
  "trace_id": "req-test-123",
  "current_phase": "analysis",
  "retry_counts": {
    "extraction": 1,
    "analysis": 0
  },
  "max_retries": {
    "extraction": 3,
    "analysis": 2
  },
  "error_count": 0,
  "force_fail": false
}
✅ PASS


## 3. 초기 상태 - Extraction (trace_id 자동 생성)

In [5]:
print("="*70)
print("📋 Test 3: 초기 상태 -> parallel_extraction (trace_id 자동 생성)")
print("="*70)

initial_state = {
    "content": "테스트 스토리",
    "extraction_done": False,
    "analysis_done": False,
    "validation_done": False,
    "errors": []
}

result = run_async(supervisor_node(initial_state))

assert "trace_id" in result, "trace_id가 자동 생성되어야 함"
assert "supervisor_state" in result, "supervisor_state가 있어야 함"
print(f"Generated trace_id: {result['trace_id']}")
print(json.dumps(result['supervisor_state'], indent=2))
print("✅ PASS")

📋 Test 3: 초기 상태 -> parallel_extraction (trace_id 자동 생성)
[req-20251228-124526-fe22794e] New request started
[unknown] Phase Status:
  - Extraction: done=False, chars=0, events=0
  - Analysis: done=False, rels=0, consistency=0
  - Validation: done=False, score=0, action=pending
  - Retries: extraction=0/3
  - Errors: 0
[unknown] -> parallel_extraction (initial)
Generated trace_id: req-20251228-124526-fe22794e
{
  "trace_id": "req-20251228-124526-fe22794e",
  "current_phase": "extraction",
  "retry_counts": {
    "extraction": 0,
    "analysis": 0
  },
  "max_retries": {
    "extraction": 3,
    "analysis": 2
  },
  "error_count": 0,
  "force_fail": false
}
✅ PASS


## 4. Happy Path - 전체 파이프라인

In [6]:
print("="*70)
print("📋 Test 4: Happy Path 전체 파이프라인")
print("="*70)

trace_id = generate_trace_id()
print(f"Trace ID: {trace_id}\n")

# Phase 1: Initial
state = {"trace_id": trace_id}
result = supervisor_router(state)
print(f"Step 1: {result}")
assert result == "parallel_extraction"

# Phase 2: After extraction
state["extraction_done"] = True
state["extracted_characters"] = [{"name": "서진"}]
result = supervisor_router(state)
print(f"Step 2: {result}")
assert result == "parallel_analysis"

# Phase 3: After analysis
state["analysis_done"] = True
result = supervisor_router(state)
print(f"Step 3: {result}")
assert result == "validation"

# Phase 4: After validation (success)
state["validation_done"] = True
state["validation_result"] = {"quality_score": 100, "action": "approve"}
result = supervisor_router(state)
print(f"Step 4: {result}")
assert result == "__end__"

print("\n✅ PASS - Happy Path Complete")

📋 Test 4: Happy Path 전체 파이프라인
Trace ID: req-20251228-124526-544179c5

[req-20251228-124526-544179c5] Phase Status:
  - Extraction: done=False, chars=0, events=0
  - Analysis: done=False, rels=0, consistency=0
  - Validation: done=False, score=0, action=pending
  - Retries: extraction=0/3
  - Errors: 0
[req-20251228-124526-544179c5] -> parallel_extraction (initial)
Step 1: parallel_extraction
[req-20251228-124526-544179c5] Phase Status:
  - Extraction: done=True, chars=1, events=0
  - Analysis: done=False, rels=0, consistency=0
  - Validation: done=False, score=0, action=pending
  - Retries: extraction=0/3
  - Errors: 0
[req-20251228-124526-544179c5] -> parallel_analysis
Step 2: parallel_analysis
[req-20251228-124526-544179c5] Phase Status:
  - Extraction: done=True, chars=1, events=0
  - Analysis: done=True, rels=0, consistency=0
  - Validation: done=False, score=0, action=pending
  - Retries: extraction=0/3
  - Errors: 0
[req-20251228-124526-544179c5] -> validation
Step 3: validation


## 5. 피드백 루프 - Retry with trace_id

In [7]:
print("="*70)
print("📋 Test 5: 피드백 루프 (Validation 실패 -> 재시도)")
print("="*70)

trace_id = generate_trace_id()

validation_failed = {
    "trace_id": trace_id,
    "extraction_done": True,
    "analysis_done": True,
    "validation_done": True,
    "validation_result": {
        "quality_score": 30,
        "action": "retry_extraction"
    },
    "extraction_retry_count": 0  # 첫 재시도
}

result = supervisor_router(validation_failed)
print(f"\nRouting: {result}")
assert result == "parallel_extraction"

# supervisor_node로 재시도 카운트 확인
node_result = run_async(supervisor_node(validation_failed))
print(f"New retry count: {node_result.get('extraction_retry_count')}")
assert node_result["extraction_retry_count"] == 1
print("✅ PASS")

📋 Test 5: 피드백 루프 (Validation 실패 -> 재시도)
[req-20251228-124526-7aeed172] Phase Status:
  - Extraction: done=True, chars=0, events=0
  - Analysis: done=True, rels=0, consistency=0
  - Validation: done=True, score=30, action=retry_extraction
  - Retries: extraction=0/3
  - Errors: 0
[req-20251228-124526-7aeed172] Validation requested retry_extraction (attempt 1/3)
[req-20251228-124526-7aeed172] -> parallel_extraction (retry: validation_rejected)

Routing: parallel_extraction
[req-20251228-124526-7aeed172] Phase Status:
  - Extraction: done=True, chars=0, events=0
  - Analysis: done=True, rels=0, consistency=0
  - Validation: done=True, score=30, action=retry_extraction
  - Retries: extraction=0/3
  - Errors: 0
[req-20251228-124526-7aeed172] Validation requested retry_extraction (attempt 1/3)
[req-20251228-124526-7aeed172] -> parallel_extraction (retry: validation_rejected)
[req-20251228-124526-7aeed172] Resetting extraction for retry #1
New retry count: 1
✅ PASS


## 6. 최대 재시도 초과 -> Human Review

In [8]:
print("="*70)
print(f"📋 Test 6: 최대 재시도 초과 (max={MAX_EXTRACTION_RETRIES}) -> human_review")
print("="*70)

trace_id = generate_trace_id()

max_retries_state = {
    "trace_id": trace_id,
    "extraction_done": True,
    "analysis_done": True,
    "validation_done": True,
    "validation_result": {"quality_score": 30, "action": "retry_extraction"},
    "extraction_retry_count": MAX_EXTRACTION_RETRIES  # 최대 도달
}

result = supervisor_router(max_retries_state)
print(f"\nRouting: {result}")
assert result == "human_review", "최대 재시도 초과 시 human_review로 라우팅되어야 함"
print("✅ PASS")

📋 Test 6: 최대 재시도 초과 (max=3) -> human_review
[req-20251228-124526-1d0b2ecd] Phase Status:
  - Extraction: done=True, chars=0, events=0
  - Analysis: done=True, rels=0, consistency=0
  - Validation: done=True, score=30, action=retry_extraction
  - Retries: extraction=3/3
  - Errors: 0
[req-20251228-124526-1d0b2ecd] Max extraction retries (3) reached -> FORCE FAIL
[req-20251228-124526-1d0b2ecd] -> human_review (max retries exceeded)

Routing: human_review
✅ PASS


## 7. Human Review Node 테스트

In [9]:
print("="*70)
print("📋 Test 7: Human Review Node")
print("="*70)

trace_id = generate_trace_id()

human_review_state = {
    "trace_id": trace_id,
    "force_fail": True,
    "force_fail_reason": "max_retries_exceeded"
}

result = run_async(human_review_node(human_review_state))
print(json.dumps(result, indent=2, ensure_ascii=False))

assert result["human_review_required"] == True
assert result["human_review_reason"] == "max_retries_exceeded"
print("\n✅ PASS")

📋 Test 7: Human Review Node
[req-20251228-124526-41f85eee] Pipeline requires human review
[req-20251228-124526-41f85eee] Reason: max_retries_exceeded
{
  "human_review_required": true,
  "human_review_reason": "max_retries_exceeded",
  "validation_done": true,
  "messages": [
    {
      "role": "human_review",
      "content": "Pipeline requires human intervention. Reason: max_retries_exceeded",
      "trace_id": "req-20251228-124526-41f85eee"
    }
  ]
}

✅ PASS


## 8. Routing Summary with trace_id

In [10]:
print("="*70)
print("📤 Routing Summary")
print("="*70)

full_state = {
    "trace_id": generate_trace_id(),
    "extraction_done": True,
    "analysis_done": True,
    "validation_done": True,
    "extracted_characters": [{"name": "서진"}, {"name": "하나"}],
    "extracted_events": [{"event_id": "E001"}],
    "relationship_graph": {"relationships": [{"source": "서진", "target": "하나"}]},
    "consistency_report": {"overall_score": 85},
    "validation_result": {"quality_score": 100, "action": "approve"}
}

summary = get_routing_summary(full_state)
print(json.dumps(summary, ensure_ascii=False, indent=2))

📤 Routing Summary
[req-20251228-124526-68245fcd] Phase Status:
  - Extraction: done=True, chars=2, events=1
  - Analysis: done=True, rels=1, consistency=85
  - Validation: done=True, score=100, action=approve
  - Retries: extraction=0/3
  - Errors: 0
[req-20251228-124526-68245fcd] -> __end__ (all done)
{
  "trace_id": "req-20251228-124526-68245fcd",
  "supervisor_state": {
    "trace_id": "req-20251228-124526-68245fcd",
    "current_phase": "complete",
    "retry_counts": {
      "extraction": 0,
      "analysis": 0
    },
    "max_retries": {
      "extraction": 3,
      "analysis": 2
    },
    "error_count": 0,
    "force_fail": false
  },
  "phase_status": {
    "extraction": {
      "done": true,
      "characters": 2,
      "events": 1,
      "settings": 0,
      "dialogues": false,
      "emotions": false
    },
    "analysis": {
      "done": true,
      "relationships": 1,
      "consistency_score": 85,
      "plot_beats": 0
    },
    "validation": {
      "done": true,
     

## 9. Spring Boot Callback 데이터 구조

In [11]:
print("="*70)
print("📤 Spring Boot Callback 데이터 구조")
print("="*70)

callback_data = {
    "trace_id": full_state["trace_id"],
    "supervisor_state": get_supervisor_state(full_state),
    "validation_result": full_state["validation_result"]
}

print(json.dumps(callback_data, ensure_ascii=False, indent=2))

📤 Spring Boot Callback 데이터 구조
{
  "trace_id": "req-20251228-124526-68245fcd",
  "supervisor_state": {
    "trace_id": "req-20251228-124526-68245fcd",
    "current_phase": "complete",
    "retry_counts": {
      "extraction": 0,
      "analysis": 0
    },
    "max_retries": {
      "extraction": 3,
      "analysis": 2
    },
    "error_count": 0,
    "force_fail": false
  },
  "validation_result": {
    "quality_score": 100,
    "action": "approve"
  }
}
